# NB-01 | Benchmark: Baseline 1 — Pure Base

Оценка **Anima Base + Turbo LoRA** без какой-либо стилевой LoRA.  
Это нижняя граница — точка отсчёта для всех остальных бейзлайнов.

**Метрики:** KID, CLIP Score, LPIPS  
**MLflow:** логирует модель, инференс-параметры, метрики, sample grid

In [1]:
import os, json, struct, hashlib
from pathlib import Path
import torch
from PIL import Image
import numpy as np
import mlflow
from torchmetrics.image.kid import KernelInceptionDistance
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
import open_clip
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'MLflow version: {mlflow.__version__}')

e:\Projects\Auto-VideoGame-Assets-Pipeline\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu
MLflow version: 3.14.0


## Конфигурация

Директории с картинками и параметры инференса.

In [2]:
BASELINE_ID   = 'baseline_1'
BASELINE_DESC = 'Anima Base + Turbo LoRA only (no style LoRA)'

GENERATED_DIR  = Path('generated/baseline_1')
REFERENCE_DIR  = Path('reference/characters')

MLFLOW_URI        = 'http://127.0.0.1:5000'
EXPERIMENT_NAME   = 'anima_baseline_benchmarks'

# Параметры модели
MODEL_PARAMS = {
    'model.name':          'anima_base',
    'model.family':        'cosmos2_dit',
    'model.type':          'base_only',
    'model.diffusion':     'anima-base-v1.0.safetensors',
    'model.text_encoder':  'qwen_3_06b_base.safetensors',
    'model.vae':           'qwen_image_vae.safetensors',
    'model.lora_stack':    'turbo_only',
    'model.trigger':       'none',
}

# Параметры инференса
INFER_PARAMS = {
    'infer.baseline_id':      BASELINE_ID,
    'infer.sampler':          'euler_ancestral',
    'infer.scheduler':        'normal',
    'infer.steps':            12,
    'infer.cfg':              2.0,
    'infer.width':            512,
    'infer.height':           512,
    'infer.turbo_lora_name':  'Anima_test/anima-turbo-lora-v0.2.safetensors',
    'infer.turbo_strength':   0.7,
    'infer.style_lora_name':  'none',
    'infer.style_strength':   0.0,
    'infer.llm_expansion':    False,
    'infer.seed_policy':      'idx*7919',
    'infer.prompt_set':       'char_prompts_v1',
    'infer.n_images':         100,
    'infer.positive_prefix':  'masterpiece, best quality, score_9, score_8, score_7',
    'infer.negative_prefix':  'worst quality, low quality, artist name, watermark',
}

# Параметры датасетов
DATA_PARAMS = {
    'data.generated_dir':    str(GENERATED_DIR),
    'data.reference_dir':    str(REFERENCE_DIR),
    'data.reference_type':   'character_splash_art',
    'data.generated_source': 'nb_00_baseline_1',
}

print('Config loaded ✓')
print(f'Generated: {GENERATED_DIR}')
print(f'Reference: {REFERENCE_DIR}')

Config loaded ✓
Generated: generated\baseline_1
Reference: reference\characters


## Вспомогательные функции

Загрузка изображений, сетка для артефакта, чтение metadata LoRA.

In [3]:
def load_images_as_tensors(folder: Path, size=(299, 299), limit=None):
    """Загружает PNG из папки как uint8 тензор [N, C, H, W] для KID."""
    files = sorted(folder.glob('*.png'))
    if limit: files = files[:limit]
    tensors = []
    for p in files:
        img = Image.open(p).convert('RGB').resize(size)
        t = torch.tensor(np.array(img)).permute(2, 0, 1)  # CHW
        tensors.append(t)
    return torch.stack(tensors)  # [N, 3, H, W]


def load_images_pil(folder: Path, size=None, limit=None):
    files = sorted(folder.glob('*.png'))
    if limit: files = files[:limit]
    imgs = []
    for p in files:
        img = Image.open(p).convert('RGB')
        if size: img = img.resize(size)
        imgs.append(img)
    return imgs


def make_grid(images, n=8, thumb=128):
    """Собирает сетку из первых n изображений."""
    imgs = [img.resize((thumb, thumb)) for img in images[:n]]
    row = Image.new('RGB', (thumb * len(imgs), thumb), (20, 20, 20))
    for i, im in enumerate(imgs):
        row.paste(im, (thumb * i, 0))
    return row


def file_sha256(path: Path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()[:16]


def read_safetensors_metadata(path: Path):
    """Читает JSON-метаданные из заголовка safetensors файла."""
    try:
        with open(path, 'rb') as f:
            length_bytes = f.read(8)
            header_length = struct.unpack('<Q', length_bytes)[0]
            header_json = f.read(header_length).decode('utf-8')
            header = json.loads(header_json)
            return header.get('__metadata__', {})
    except Exception as e:
        return {'error': str(e)}


print('Helpers ready ✓')

Helpers ready ✓


## Вычисление метрик

Три метрики:
- **KID** (Kernel Inception Distance): распределение стиля vs reference
- **CLIP Score**: text-image alignment
- **LPIPS**: разнообразие внутри батча (защита от mode collapse)

In [4]:
def compute_kid(gen_dir: Path, ref_dir: Path, device=DEVICE, subset=50):
    """KID между сгенерированными и референсными изображениями."""
    print('Computing KID...')
    kid = KernelInceptionDistance(subset_size=subset, normalize=True).to(device)
    gen_imgs = load_images_as_tensors(gen_dir).to(device)
    ref_imgs = load_images_as_tensors(ref_dir).to(device)
    kid.update(ref_imgs, real=True)
    kid.update(gen_imgs, real=False)
    mean, std = kid.compute()
    return float(mean.cpu()), float(std.cpu())


def compute_clip_score(gen_dir: Path, prompts: list, device=DEVICE):
    """Cosine similarity между изображениями и их промптами через CLIP."""
    print('Computing CLIP Score...')
    model, _, preprocess = open_clip.create_model_and_transforms(
        'ViT-B-32', pretrained='openai'
    )
    model = model.to(device).eval()
    tokenizer = open_clip.get_tokenizer('ViT-B-32')

    files = sorted(gen_dir.glob('*.png'))
    scores = []
    for i, p in enumerate(tqdm(files, desc='CLIP')):
        if i >= len(prompts): break
        img = preprocess(Image.open(p).convert('RGB')).unsqueeze(0).to(device)
        text = tokenizer([prompts[i]]).to(device)
        with torch.no_grad():
            img_f  = model.encode_image(img)
            txt_f  = model.encode_text(text)
            img_f  = img_f / img_f.norm(dim=-1, keepdim=True)
            txt_f  = txt_f / txt_f.norm(dim=-1, keepdim=True)
            scores.append(float((img_f * txt_f).sum().cpu()))
    return float(np.mean(scores))


def compute_lpips_diversity(gen_dir: Path, n_pairs=200, device=DEVICE):
    """Среднее LPIPS между случайными парами — мера разнообразия батча."""
    print('Computing LPIPS diversity...')
    lpips_fn = LearnedPerceptualImagePatchSimilarity(net_type='vgg', normalize=True).to(device)
    files = sorted(gen_dir.glob('*.png'))
    idx = np.random.default_rng(42).integers(0, len(files), size=(n_pairs, 2))
    scores = []
    for a, b in tqdm(idx, desc='LPIPS'):
        if a == b: continue
        def load_t(p):
            arr = np.array(Image.open(p).convert('RGB').resize((256, 256))).astype(np.float32) / 255.
            return torch.tensor(arr).permute(2,0,1).unsqueeze(0).to(device)
        with torch.no_grad():
            scores.append(float(lpips_fn(load_t(files[a]), load_t(files[b])).cpu()))
    return float(np.mean(scores))


print('Metric functions ready ✓')

Metric functions ready ✓


## Загрузка промптов для CLIP Score

Промпты должны совпадать с теми, что использовались при генерации в NB-00.

In [5]:
CHARACTER_PROMPTS_BASE = [
    '1girl, solo, long hair, blue eyes, smile, outdoors, sky',
    '1girl, solo, short hair, red eyes, serious, city background',
    '1girl, solo, twin tails, green eyes, school uniform, outdoors',
    '1girl, solo, white hair, purple eyes, simple background',
    '1girl, solo, ponytail, brown eyes, jacket, forest',
    '1boy, solo, short hair, blue eyes, armor, standing',
    '1girl, solo, black hair, dress, indoors',
    '1girl, solo, cat ears, pink hair, happy, outdoors',
    '1boy, solo, messy hair, red scarf, city',
    '1girl, solo, silver hair, golden eyes, cape, sky background',
]
PROMPTS_100 = [CHARACTER_PROMPTS_BASE[i % len(CHARACTER_PROMPTS_BASE)] for i in range(100)]
print(f'Prompts loaded: {len(PROMPTS_100)}')

Prompts loaded: 100


## MLflow Run — Логирование всего

Здесь происходит:
1. Создание run с правильным именем
2. Логирование model params, infer params, data params
3. Вычисление метрик
4. Логирование метрик
5. Сохранение sample grid как artifact
6. Сохранение summary JSON

In [6]:
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

run_name = f'eval_{BASELINE_ID}_turbo_only_cfg{INFER_PARAMS["infer.cfg"]}_steps{INFER_PARAMS["infer.steps"]}'

with mlflow.start_run(run_name=run_name) as run:
    print(f'Run ID: {run.info.run_id}')

    # ── Tags ────────────────────────────────────────────────────────
    mlflow.set_tags({
        'evaluation_type':             'benchmark',
        'training_logged_retroactively': 'false',
        'family':                       'character',
        'baseline_id':                  BASELINE_ID,
        'environment':                  'local_rtx4060ti_16gb',
        'framework':                    'comfyui',
    })

    # ── Parameters ─────────────────────────────────────────────────
    mlflow.log_params(MODEL_PARAMS)
    mlflow.log_params(INFER_PARAMS)
    mlflow.log_params(DATA_PARAMS)

    # ── Compute metrics ────────────────────────────────────────────
    kid_mean, kid_std = compute_kid(GENERATED_DIR, REFERENCE_DIR)
    clip_score        = compute_clip_score(GENERATED_DIR, PROMPTS_100)
    lpips_div         = compute_lpips_diversity(GENERATED_DIR)

    n_gen = len(list(GENERATED_DIR.glob('*.png')))
    n_ref = len(list(REFERENCE_DIR.glob('*.png')))

    mlflow.log_metrics({
        'eval.kid_mean':        kid_mean,
        'eval.kid_std':         kid_std,
        'eval.clip_score':      clip_score,
        'eval.lpips_diversity': lpips_div,
        'eval.n_generated':     float(n_gen),
        'eval.n_reference':     float(n_ref),
    })

    # ── Sample grid artifact ───────────────────────────────────────
    pil_imgs = load_images_pil(GENERATED_DIR, limit=8)
    grid = make_grid(pil_imgs)
    Path('tmp').mkdir(parents=True, exist_ok=True)
    grid_path = f'tmp/{BASELINE_ID}_sample_grid.png'
    grid.save(grid_path)
    mlflow.log_artifact(grid_path, artifact_path='samples')

    # ── Summary JSON artifact ──────────────────────────────────────
    summary = {
        'baseline_id': BASELINE_ID,
        'run_id': run.info.run_id,
        'metrics': {
            'kid_mean':        round(kid_mean, 6),
            'kid_std':         round(kid_std, 6),
            'clip_score':      round(clip_score, 4),
            'lpips_diversity': round(lpips_div, 4),
        },
        'model': MODEL_PARAMS,
        'infer': INFER_PARAMS,
    }
    summary_path = f'tmp/{BASELINE_ID}_summary.json'
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    mlflow.log_artifact(summary_path, artifact_path='reports')

    print(f'\n=== Baseline 1 Results ===')
    print(f'KID mean:         {kid_mean:.6f} ± {kid_std:.6f}')
    print(f'CLIP Score:       {clip_score:.4f}')
    print(f'LPIPS Diversity:  {lpips_div:.4f}')
    print(f'Run logged to MLflow ✓')

Run ID: d60f95d277294fa9942cc9257b7290f6
Computing KID...


e:\Projects\Auto-VideoGame-Assets-Pipeline\venv\lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: Metric `Kernel Inception Distance` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to C:\Users\b8914/.cache\torch\hub\checkpoints\weights-inception-2015-12-05-6726825d.pth
100%|██████████| 91.2M/91.2M [00:12<00:00, 7.50MB/s]


Computing CLIP Score...


e:\Projects\Auto-VideoGame-Assets-Pipeline\venv\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\b8914\.cache\huggingface\hub\models--timm--vit_base_patch32_clip_224.openai. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
e:\Projects\Auto-VideoGame-Assets-Pipeline\venv\lib\site-packages\open_clip\factory

Computing LPIPS diversity...
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to C:\Users\b8914/.cache\torch\hub\checkpoints\vgg16-397923af.pth


100%|██████████| 528M/528M [01:05<00:00, 8.49MB/s] 
LPIPS: 100%|██████████| 200/200 [01:03<00:00,  3.15it/s]



=== Baseline 1 Results ===
KID mean:         0.073195 ± 0.006374
CLIP Score:       0.3042
LPIPS Diversity:  0.6914
Run logged to MLflow ✓
🏃 View run eval_baseline_1_turbo_only_cfg2.0_steps12 at: http://127.0.0.1:5000/#/experiments/1/runs/d60f95d277294fa9942cc9257b7290f6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
